# Day 5 — LLM Deployment & Production Systems
## Industrial AI & LLM Training Program

**Session:** Day 5 of 5 — LLM Deployment & Production Systems
**Time:** 20:00–22:30 WIB
**Objective:** Deploy and optimize LLM systems for production — local inference, quantization, cost monitoring, fine-tuning, and end-to-end pipeline.

---

### How to Use This Notebook
1. Run cells top to bottom — each section builds on the previous
2. If you don't have an API key, **mock mode** runs automatically
3. If you don't have a GPU, fine-tuning cells run in **simulation mode**
4. All outputs are deterministic in mock/simulation mode for reproducibility

In [ ]:
# CELL 0-A: Environment setup — load .env, detect provider
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    env_path = Path('.env')
    if env_path.exists():
        load_dotenv(env_path)
        print("✓ .env loaded")
    else:
        print("⚠ No .env file found — will use mock mode")
except ImportError:
    print("ℹ python-dotenv not installed — reading env vars directly")

# Detect provider
ANTHROPIC_KEY = os.getenv('ANTHROPIC_API_KEY', '')
OPENAI_KEY = os.getenv('OPENAI_API_KEY', '')

if ANTHROPIC_KEY and not ANTHROPIC_KEY.startswith('your-'):
    PROVIDER = 'anthropic'
    print(f"✓ Provider: Anthropic (key: ...{ANTHROPIC_KEY[-8:]})")
elif OPENAI_KEY and not OPENAI_KEY.startswith('your-'):
    PROVIDER = 'openai'
    print(f"✓ Provider: OpenAI (key: ...{OPENAI_KEY[-8:]})")
else:
    PROVIDER = 'mock'
    print("ℹ Provider: Mock (no API key — all LLM calls simulated)")

# Check Ollama availability
import requests
try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=2)
    OLLAMA_AVAILABLE = resp.status_code == 200
    if OLLAMA_AVAILABLE:
        models = [m['name'] for m in resp.json().get('models', [])]
        print(f"✓ Ollama: available | Models: {models if models else 'none pulled'}")
    else:
        OLLAMA_AVAILABLE = False
        print("ℹ Ollama: running but unexpected response")
except Exception:
    OLLAMA_AVAILABLE = False
    print("ℹ Ollama: not running (will use mock)")

print(f"\nSetup complete. PROVIDER={PROVIDER}, OLLAMA_AVAILABLE={OLLAMA_AVAILABLE}")

In [ ]:
# CELL 0-B: Imports + unified LLM caller
import json, time, random, warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import psutil
import platform

warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

# Pricing (USD per 1M tokens)
PRICING = {
    'anthropic': {'claude-haiku-4-5-20251001': {'in': 0.25, 'out': 1.25},
                  'claude-sonnet-4-6': {'in': 3.00, 'out': 15.00}},
    'openai':    {'gpt-3.5-turbo': {'in': 0.50, 'out': 1.50},
                  'gpt-4o': {'in': 5.00, 'out': 15.00}},
    'mock':      {'mock': {'in': 0.00, 'out': 0.00}},
}

GPU_AVAILABLE = False

def call_llm(prompt, system="You are an industrial support AI.", max_tokens=256):
    """Unified LLM caller with Anthropic/OpenAI/Mock fallback."""
    if PROVIDER == 'anthropic':
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
        msg = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return msg.content[0].text, msg.usage.input_tokens, msg.usage.output_tokens
    elif PROVIDER == 'openai':
        import openai
        client = openai.OpenAI(api_key=OPENAI_KEY)
        resp = client.chat.completions.create(
            model="gpt-3.5-turbo", max_tokens=max_tokens,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": prompt}]
        )
        u = resp.usage
        return resp.choices[0].message.content, u.prompt_tokens, u.completion_tokens
    else:
        # Deterministic mock responses
        cat_hints = {'mechanical': 'Mechanical', 'safety': 'Safety',
                     'normal': 'Normal', 'complex': 'Complex'}
        cat = 'Mechanical'
        for k, v in cat_hints.items():
            if k in prompt.lower():
                cat = v
                break
        response = f"Category: {cat}\nPriority: High\nAction: Schedule inspection within 24 hours."
        tokens_in = max(10, len(prompt.split()) // 2)
        tokens_out = len(response.split())
        return response, tokens_in, tokens_out

print("✓ Imports complete")
print(f"  pandas {pd.__version__} | psutil {psutil.__version__}")
print(f"  Provider: {PROVIDER} | Mock fallback: always available")

---
## Section 1 — Hardware & Memory Analysis

Before deploying any model, size your hardware.
This section detects your system specs and calculates memory requirements
for different model sizes and quantization levels.

In [ ]:
# CELL 1-A: Hardware detection
print("=" * 50)
print("SYSTEM HARDWARE REPORT")
print("=" * 50)

cpu_count = psutil.cpu_count(logical=True)
cpu_physical = psutil.cpu_count(logical=False)
cpu_freq = psutil.cpu_freq()

print(f"\n📦 CPU:")
print(f"  Platform:       {platform.system()} {platform.machine()}")
print(f"  Physical cores: {cpu_physical}")
print(f"  Logical cores:  {cpu_count}")
if cpu_freq:
    print(f"  Frequency:      {cpu_freq.current:.0f} MHz")

ram = psutil.virtual_memory()
print(f"\n💾 RAM:")
print(f"  Total:     {ram.total / (1024**3):.1f} GB")
print(f"  Available: {ram.available / (1024**3):.1f} GB")
print(f"  Used:      {ram.used / (1024**3):.1f} GB ({ram.percent:.1f}%)")

disk = psutil.disk_usage('/')
print(f"\n💿 Disk (root):")
print(f"  Total:  {disk.total / (1024**3):.0f} GB  |  Free: {disk.free / (1024**3):.0f} GB")

# GPU detection
try:
    import subprocess
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, timeout=5
    )
    if result.returncode == 0:
        print("\n🎮 GPU:")
        for i, line in enumerate(result.stdout.strip().split('\n')):
            name, total, free = [x.strip() for x in line.split(',')]
            print(f"  GPU {i}: {name}")
            print(f"    VRAM: {float(total)/1024:.1f} GB total | {float(free)/1024:.1f} GB free")
        GPU_AVAILABLE = True
    else:
        print("\n🎮 GPU: NVIDIA GPU not detected")
        GPU_AVAILABLE = False
except Exception:
    print("\n🎮 GPU: nvidia-smi not available")
    GPU_AVAILABLE = False

print(f"\n✓ Hardware check complete | GPU_AVAILABLE={GPU_AVAILABLE}")

In [ ]:
# CELL 1-B: Model memory requirements + visualization
def model_memory_gb(params_b, dtype):
    bpp = {'FP32': 4, 'FP16': 2, 'BF16': 2, 'INT8': 1, 'INT4': 0.5}
    return round(params_b * 1e9 * bpp[dtype] / (1024**3), 1)

models = [('3B', 3), ('7B', 7), ('13B', 13), ('70B', 70)]
dtypes = ['FP32', 'FP16', 'INT8', 'INT4']

rows = []
for name, p in models:
    row = {'Model': name}
    for d in dtypes:
        row[d] = f"{model_memory_gb(p, d)} GB"
    rows.append(row)

df_mem = pd.DataFrame(rows).set_index('Model')
print("Model Memory Requirements")
print("=" * 50)
print(df_mem.to_string())
print()

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Your RAM: {ram_gb:.1f} GB")
if ram_gb >= 32:
    print("  ✓ Can run 13B INT4 locally via Ollama")
elif ram_gb >= 16:
    print("  ✓ Can run 7B INT4 locally via Ollama")
else:
    print("  ✓ Can run 3B INT4 locally via Ollama")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))
w = 0.2
colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
for i, (d, c) in enumerate(zip(dtypes, colors)):
    vals = [model_memory_gb(p, d) for _, p in models]
    ax.bar(x + i*w, vals, w, label=d, color=c, alpha=0.82)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels([m[0] for m in models])
ax.set_xlabel('Model Size')
ax.set_ylabel('Memory (GB)')
ax.set_title('Model Memory Requirements by Quantization Level')
ax.legend()
ax.axhline(16, color='purple', ls='--', alpha=0.4, label='16 GB')
ax.axhline(40, color='gray', ls='--', alpha=0.4, label='40 GB')
plt.tight_layout()
plt.show()
print("✓ Sweet spot: 7B INT4 = 3.5 GB — fits on most consumer hardware")

---
## Section 2 — Ollama: Local Model Management

Ollama makes it easy to run LLMs locally — no GPU required for small models.
We test connectivity, send a real prompt, and compare local vs cloud latency.

In [ ]:
# CELL 2-A: Ollama health check and inference
def ollama_generate(prompt, model='llama3.2', timeout=30):
    try:
        r = requests.post('http://localhost:11434/api/generate',
                          json={'model': model, 'prompt': prompt, 'stream': False},
                          timeout=timeout)
        r.raise_for_status()
        d = r.json()
        return d['response'], d.get('eval_count', 0), d.get('prompt_eval_count', 0)
    except Exception:
        return None, 0, 0

test_prompt = "Classify in one line: 'Pump P-101 vibration 12mm/s, bearing temp 85°C. Maintenance needed.'"

print("Ollama Connectivity Test")
print("=" * 45)

if OLLAMA_AVAILABLE:
    start = time.time()
    response, eval_t, prompt_t = ollama_generate(test_prompt)
    elapsed = time.time() - start
    if response:
        print(f"✓ Response time: {elapsed:.2f}s | Tokens: {prompt_t} in / {eval_t} out")
        print(f"\nResponse:\n{response[:300]}")
    else:
        print("⚠ Ollama running but inference failed.")
        print("  Try: ollama pull llama3.2")
else:
    print("ℹ Ollama not available — showing mock response")
    print("\nMock response:")
    print("  Category: Mechanical | Priority: Critical | Action: Emergency inspection required")
    print("\nTo install Ollama:")
    print("  curl -fsSL https://ollama.ai/install.sh | sh")
    print("  ollama pull llama3.2")

In [ ]:
# CELL 2-B: Latency comparison — local Ollama vs cloud API
prompts = [
    "Classify: Motor M-202 overheating, ambient 45°C",
    "Classify: Conveyor belt CV-05 jammed, foreign object",
    "Classify: Pressure gauge PG-301 reading zero",
]

results = []
print(f"{'Prompt':<45} {'Ollama (ms)':>12} {'Cloud (ms)':>12}")
print("-" * 72)

for p in prompts:
    # Ollama
    if OLLAMA_AVAILABLE:
        t0 = time.time()
        ollama_generate(p)
        ollama_ms = int((time.time()-t0)*1000)
    else:
        ollama_ms = random.randint(700, 1600)  # simulated

    # Cloud API
    if PROVIDER != 'mock':
        t0 = time.time()
        call_llm(p)
        cloud_ms = int((time.time()-t0)*1000)
    else:
        cloud_ms = random.randint(250, 700)  # simulated

    results.append({'ollama_ms': ollama_ms, 'cloud_ms': cloud_ms})
    print(f"{p[:44]:<45} {ollama_ms:>12} {cloud_ms:>12}")

avg_o = np.mean([r['ollama_ms'] for r in results])
avg_c = np.mean([r['cloud_ms'] for r in results])
print(f"\nAvg: Ollama={avg_o:.0f}ms  Cloud={avg_c:.0f}ms")
print(f"Cost: Ollama=$0.00/req  Cloud=~$0.0001/req")
print("\nKey insight: local is slower per request but zero marginal cost → better at scale")

---
## Section 3 — Model Quantization Analysis

Quantization reduces model size at a small quality cost.
We compute metrics for each quantization level and visualize the trade-offs.

In [ ]:
# CELL 3-A: Quantization metrics
quant_data = [
    {'level': 'FP32',         'bits': 32, 'mem_7b': 28.0, 'rel_speed': 0.5,  'quality': 1.000, 'use_case': 'Research only'},
    {'level': 'FP16 / BF16',  'bits': 16, 'mem_7b': 14.0, 'rel_speed': 1.0,  'quality': 0.999, 'use_case': 'vLLM baseline, GPU serving'},
    {'level': 'INT8 (Q8_0)',   'bits':  8, 'mem_7b':  7.0, 'rel_speed': 1.4,  'quality': 0.990, 'use_case': 'Production: quality-first'},
    {'level': 'INT4 (Q4_K_M)', 'bits':  4, 'mem_7b':  3.5, 'rel_speed': 1.8,  'quality': 0.960, 'use_case': 'Production: balanced ★ recommended'},
    {'level': 'INT2 (Q2_K)',   'bits':  2, 'mem_7b':  1.75,'rel_speed': 2.1,  'quality': 0.870, 'use_case': 'Edge: memory-critical only'},
]

df_q = pd.DataFrame(quant_data)
df_q['quality_%'] = (df_q['quality']*100).map('{:.1f}%'.format)
df_q['speed'] = df_q['rel_speed'].map('{:.1f}×'.format)

print("Quantization Comparison (7B model)")
print("=" * 72)
print(df_q[['level','mem_7b','speed','quality_%','use_case']].to_string(index=False))
print("\n★ Recommended for most production use cases: INT4 (Q4_K_M)")

In [ ]:
# CELL 3-B: Memory vs Quality scatter + Decision heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors_q = ['#e74c3c','#e67e22','#3498db','#2ecc71','#9b59b6']
for i, row in df_q.iterrows():
    ax1.scatter(row['mem_7b'], row['quality']*100, s=220, color=colors_q[i],
                zorder=5, edgecolors='black', lw=0.8)
    ax1.annotate(row['level'], (row['mem_7b'], row['quality']*100),
                 textcoords="offset points", xytext=(7, 4), fontsize=9)
ax1.set_xlabel('Memory (GB) — 7B Model')
ax1.set_ylabel('Quality Retention (%)')
ax1.set_title('Memory vs Quality Trade-off')
ax1.axvspan(2, 5, alpha=0.08, color='green', label='Sweet spot (INT4)')
ax1.legend(loc='lower right')

# Heatmap
criteria = ['Memory\nEfficiency', 'Speed', 'Quality', 'Compatibility']
levels = ['FP16', 'INT8', 'INT4', 'INT2']
scores = np.array([[3,4,5,5],[4,4,5,4],[5,5,4,5],[5,5,2,3]])
im = ax2.imshow(scores, cmap='RdYlGn', vmin=1, vmax=5, aspect='auto')
ax2.set_xticks(range(len(criteria))); ax2.set_xticklabels(criteria, fontsize=9)
ax2.set_yticks(range(len(levels))); ax2.set_yticklabels(levels)
ax2.set_title('Decision Matrix (1=Poor, 5=Excellent)')
for i in range(len(levels)):
    for j in range(len(criteria)):
        ax2.text(j, i, str(scores[i][j]), ha='center', va='center', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax2, fraction=0.046)
plt.tight_layout()
plt.show()
print("✓ INT4 (Q4_K_M) wins on Memory + Speed while keeping 96% quality")

---
## Section 4 — Cost Monitoring & Token Budget

Every LLM call costs money. Track spend per request and category.

In [ ]:
# CELL 4-A: TokenBudgetTracker class
class TokenBudgetTracker:
    """Track token usage and cost across LLM requests."""
    def __init__(self, budget_usd=1.0, provider='mock', model=None):
        self.budget_usd = budget_usd
        self.provider = provider
        self.model = model or list(PRICING.get(provider, PRICING['mock']).keys())[0]
        self.records = []
        self.total_cost = 0.0
        self.total_tokens_in = 0
        self.total_tokens_out = 0

    def _price(self):
        return PRICING.get(self.provider, PRICING['mock']).get(
            self.model, {'in': 0.0, 'out': 0.0})

    def record(self, ticket_id, category, tokens_in, tokens_out, latency_ms):
        p = self._price()
        cost = (tokens_in * p['in'] + tokens_out * p['out']) / 1_000_000
        self.total_cost += cost
        self.total_tokens_in += tokens_in
        self.total_tokens_out += tokens_out
        rec = {'ticket_id': ticket_id, 'category': category,
               'tokens_in': tokens_in, 'tokens_out': tokens_out,
               'cost_usd': cost, 'latency_ms': latency_ms,
               'cumulative_cost': self.total_cost,
               'budget_pct': self.total_cost / self.budget_usd * 100}
        self.records.append(rec)
        return rec

    def summary(self):
        n = max(len(self.records), 1)
        lats = [r['latency_ms'] for r in self.records]
        return {'total_requests': len(self.records),
                'total_tokens_in': self.total_tokens_in,
                'total_tokens_out': self.total_tokens_out,
                'total_cost_usd': self.total_cost,
                'avg_cost_per_req': self.total_cost / n,
                'avg_latency_ms': sum(lats)/n if lats else 0}

    def alert(self):
        pct = self.total_cost / self.budget_usd * 100
        if pct >= 100:
            print(f"🚨 BUDGET EXCEEDED: {pct:.1f}% (${self.total_cost:.4f})")
        elif pct >= 80:
            print(f"⚠️  Budget warning: {pct:.1f}% used")

tracker = TokenBudgetTracker(budget_usd=0.10, provider=PROVIDER)
print(f"✓ TokenBudgetTracker ready | Budget: $0.10 | Provider: {PROVIDER} | Model: {tracker.model}")

In [ ]:
# CELL 4-B: Run 40-ticket classification with cost tracking
TICKETS = [
    {"id":"M-001","category":"Mechanical","text":"Pump P-101 vibration 12mm/s, bearing temp 85°C. Immediate attention."},
    {"id":"M-002","category":"Mechanical","text":"Conveyor CV-05 belt slipping. Motor load 15% above nominal."},
    {"id":"M-003","category":"Mechanical","text":"Compressor K-201 discharge pressure dropping. Valve wear suspected."},
    {"id":"M-004","category":"Mechanical","text":"Gearbox GB-07 unusual noise. Oil level low, possible seal leak."},
    {"id":"M-005","category":"Mechanical","text":"Fan FN-03 blade imbalance detected by vibration sensor VS-12."},
    {"id":"M-006","category":"Mechanical","text":"Heat exchanger HX-201 fouling. Delta-T reduced 40% from baseline."},
    {"id":"M-007","category":"Mechanical","text":"Pump P-202 cavitation noise. Flow meter FM-05 reading 0."},
    {"id":"M-008","category":"Mechanical","text":"Mixer MX-01 torque spike during startup. Coupling misalignment possible."},
    {"id":"M-009","category":"Mechanical","text":"Cooling tower CT-03 drift eliminator damage. Efficiency 60%."},
    {"id":"M-010","category":"Mechanical","text":"Steam trap ST-08 failed open. Steam loss confirmed by IR scan."},
    {"id":"S-001","category":"Safety","text":"Gas leak detected near valve V-301. H2S sensor 15 ppm. Area cleared."},
    {"id":"S-002","category":"Safety","text":"Scaffolding collapse risk Zone B. Two damaged boards found."},
    {"id":"S-003","category":"Safety","text":"Chemical spill Tank T-102. MSDS: corrosive. Containment activated."},
    {"id":"S-004","category":"Safety","text":"Fire suppression FP-01 low pressure alarm. Inspection needed."},
    {"id":"S-005","category":"Safety","text":"Worker fall protection harness inspection overdue in Area C."},
    {"id":"S-006","category":"Safety","text":"High voltage panel HV-04 door latch broken. Unauthorized access risk."},
    {"id":"S-007","category":"Safety","text":"PRV-05 leaking. Boiler B-01 overpressure risk."},
    {"id":"S-008","category":"Safety","text":"Emergency exit sign EX-07 not lit. Evacuation route compromised."},
    {"id":"S-009","category":"Safety","text":"Forklift FL-03 brake failure reported by operator. Out of service."},
    {"id":"S-010","category":"Safety","text":"Nitrogen purge line NL-09 valve open. Oxygen deficiency hazard."},
    {"id":"N-001","category":"Normal","text":"Quarterly oil change Motor M-101. PM schedule on time."},
    {"id":"N-002","category":"Normal","text":"Filter replacement FLT-03 per scheduled maintenance."},
    {"id":"N-003","category":"Normal","text":"Annual calibration of flow meter FM-07. NIST traceable."},
    {"id":"N-004","category":"Normal","text":"Lubrication route completed. All 24 points serviced."},
    {"id":"N-005","category":"Normal","text":"Belt tension check CV-02. Adjusted to 180 N. Within spec."},
    {"id":"N-006","category":"Normal","text":"Control valve CV-301 stroke test. Pass. No action required."},
    {"id":"N-007","category":"Normal","text":"Cooling water quality test. pH 7.2, TDS 450 ppm. OK."},
    {"id":"N-008","category":"Normal","text":"Weekly inspection walkthrough Area D. No anomalies found."},
    {"id":"N-009","category":"Normal","text":"Torque check on flange bolts FLG-12. All within spec."},
    {"id":"N-010","category":"Normal","text":"Greasing of coupling CP-04 completed per SOP-M-014."},
    {"id":"K-001","category":"Complex","text":"P-101 vibration elevated AND temperature rising. History shows 3 similar events this year."},
    {"id":"K-002","category":"Complex","text":"Multiple alarms: Low lube oil + high bearing temp + high vibration on turbine TU-01."},
    {"id":"K-003","category":"Complex","text":"Process yield dropped 8% after PM on reactor R-201. Root cause unclear."},
    {"id":"K-004","category":"Complex","text":"Intermittent trip on motor M-301. Occurs only at high ambient temp. Log analysis needed."},
    {"id":"K-005","category":"Complex","text":"Column C-101 pressure differential 15% above design. Flooding or fouling?"},
    {"id":"K-006","category":"Complex","text":"New equipment PMP-NEW startup. Vibration 8mm/s at 3x running speed. Resonance or defect?"},
    {"id":"K-007","category":"Complex","text":"Boiler B-02 efficiency down 6%. Could be burner, fouling, excess air, or steam trap."},
    {"id":"K-008","category":"Complex","text":"Safety system SIS-04 spurious trip 3 times this month. Sensor noise or genuine hazard?"},
    {"id":"K-009","category":"Complex","text":"Corrosion found on 5 vessels in same service. Common cause? Water ingress or chemical change?"},
    {"id":"K-010","category":"Complex","text":"After plant expansion, cooling tower CT-01 undersized. Options: upgrade, add new, or optimize?"},
]

SYS = "Classify ticket: Category:[Mechanical|Safety|Normal|Complex] Priority:[Critical|High|Medium|Low]. Reply 2 lines."

print(f"Processing {len(TICKETS)} tickets...")
for t in TICKETS:
    t0 = time.time()
    _, ti, to = call_llm(t['text'], system=SYS, max_tokens=40)
    ms = int((time.time()-t0)*1000)
    tracker.record(t['id'], t['category'], ti, to, ms)

s = tracker.summary()
tracker.alert()
print(f"\nCost Summary:")
print(f"  Requests: {s['total_requests']} | Tokens in: {s['total_tokens_in']:,} | Tokens out: {s['total_tokens_out']:,}")
print(f"  Total cost: ${s['total_cost_usd']:.4f} | Avg/req: ${s['avg_cost_per_req']:.6f} | Avg latency: {s['avg_latency_ms']:.0f}ms")

df_recs = pd.DataFrame(tracker.records)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
df_recs.groupby('category')['cost_usd'].sum().plot(kind='bar', ax=ax1, color=['#3498db','#e74c3c','#2ecc71','#f39c12'], alpha=0.85)
ax1.set_title('Cost by Category ($)'); ax1.tick_params(axis='x', rotation=30)
ax2.plot([r['cumulative_cost'] for r in tracker.records], color='#9b59b6', lw=2)
ax2.axhline(tracker.budget_usd, color='red', ls='--', label=f'Budget ${tracker.budget_usd}')
ax2.set_title('Cumulative Cost — 40 Tickets'); ax2.set_xlabel('Ticket #'); ax2.legend()
plt.tight_layout(); plt.show()

---
## Section 5 — Latency Benchmarking

Measure request latency, compute p50/p95/p99, and check SLA compliance.

In [ ]:
# CELL 5-A: Latency benchmark — 10 requests
N_BENCH = 10
latencies = []
bench_prompt = "Classify: Pump P-101 bearing temp 82°C, vibration 9mm/s. Trend increasing 48h."

print(f"Running {N_BENCH} latency benchmark requests...")
print("-" * 40)

for i in range(N_BENCH):
    t0 = time.time()
    call_llm(bench_prompt, max_tokens=50)
    ms = (time.time()-t0)*1000
    if PROVIDER == 'mock':
        ms = max(200, np.random.normal(450, 80))
    latencies.append(ms)
    status = "✓" if ms < 2000 else "⚠"
    print(f"  Request {i+1:2d}: {ms:6.0f} ms {status}")

la = np.array(latencies)
p50, p95, p99 = np.percentile(la, 50), np.percentile(la, 95), np.percentile(la, 99)
print(f"\n  Min={la.min():.0f}ms  Max={la.max():.0f}ms  Mean={la.mean():.0f}ms")
print(f"  p50={p50:.0f}ms  p95={p95:.0f}ms  p99={p99:.0f}ms")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, N_BENCH+1), latencies,
       color=['#2ecc71' if l<1000 else '#f39c12' if l<2000 else '#e74c3c' for l in latencies],
       alpha=0.8, edgecolor='black', lw=0.5)
ax.axhline(2000, color='red', ls='--', lw=1.5, label='SLA target 2000ms')
ax.axhline(p95, color='orange', ls=':', lw=1.5, label=f'p95 ({p95:.0f}ms)')
ax.set_xlabel('Request'); ax.set_ylabel('Latency (ms)')
ax.set_title(f'Request Latency — {N_BENCH} Requests'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# CELL 5-B: SLA compliance check
SLA = {'p50_ms': 1000, 'p95_ms': 2000, 'p99_ms': 5000}

print("SLA COMPLIANCE REPORT")
print("=" * 48)
print(f"{'Metric':<8} {'Target':>10} {'Actual':>10} {'Status':>10}")
print("-" * 48)

all_pass = True
for name, actual, tgt in [('p50', p50, SLA['p50_ms']),
                            ('p95', p95, SLA['p95_ms']),
                            ('p99', p99, SLA['p99_ms'])]:
    passed = actual <= tgt
    all_pass = all_pass and passed
    print(f"{name:<8} {tgt:>8}ms {actual:>8.0f}ms {'✅ PASS' if passed else '❌ FAIL':>10}")

print("-" * 48)
print(f"\nOverall: {'✅ ALL PASS — SLA MET' if all_pass else '❌ SLA BREACH — investigate'}")
within = sum(1 for l in latencies if l <= SLA['p95_ms'])
print(f"Within p95 SLA: {within}/{N_BENCH} ({within/N_BENCH*100:.0f}%)")

---
## Section 6 — Production Patterns

Three essential patterns:
1. **Cost-Aware Routing** — simple tickets → cheap model
2. **Retry with Exponential Backoff** — handle transient failures
3. **Circuit Breaker** — prevent cascade failures

In [ ]:
# CELL 6-A: Cost-aware model router
MODEL_TIERS = {
    'cheap':    {'model': 'claude-haiku-4-5-20251001', 'cost_out_per_mtok': 1.25},
    'standard': {'model': 'claude-sonnet-4-6',    'cost_out_per_mtok': 15.00},
}
COMPLEX_KEYWORDS = ['multiple','intermittent','root cause','unknown','trend',
                    'analysis','investigation','flooding','spurious','corrosion']

def route(text):
    txt = text.lower()
    score = sum(1 for kw in COMPLEX_KEYWORDS if kw in txt)
    if score >= 2 or len(text) > 200:
        return 'standard', 'complex'
    if any(kw in txt for kw in ['critical','gas leak','collapse','fire','nitrogen','h2s']):
        return 'standard', 'safety-critical'
    return 'cheap', 'simple'

results = [{'id': t['id'], 'actual': t['category'],
            'tier': route(t['text'])[0], 'complexity': route(t['text'])[1]}
           for t in TICKETS]

df_r = pd.DataFrame(results)
counts = df_r['tier'].value_counts()
print("ROUTING DISTRIBUTION:")
print(counts.to_string())

avg_out = 60
baseline = len(TICKETS) * avg_out * MODEL_TIERS['standard']['cost_out_per_mtok'] / 1e6
routed = sum(counts.get(t, 0) * avg_out * MODEL_TIERS[t]['cost_out_per_mtok']
             for t in ['cheap','standard']) / 1e6
print(f"\nCost (40 tickets, ~{avg_out} tokens out each):")
print(f"  Baseline (all Sonnet): ${baseline:.4f}")
print(f"  With routing:          ${routed:.4f}")
if baseline > 0:
    print(f"  Savings:               {(1-routed/baseline)*100:.1f}%")

In [ ]:
# CELL 6-B: Retry with exponential backoff
def with_retry(func, *args, max_attempts=3, base_s=1.0, multiplier=2.0, **kwargs):
    for attempt in range(1, max_attempts+1):
        try:
            result = func(*args, **kwargs)
            if attempt > 1:
                print(f"  ✓ Succeeded on attempt {attempt}")
            return result
        except Exception as e:
            if attempt == max_attempts:
                print(f"  ✗ All {max_attempts} attempts failed: {e}")
                raise
            delay = min(base_s * (multiplier**(attempt-1)), 60) * (0.5 + random.random()*0.5)
            print(f"  ⚠ Attempt {attempt} failed ({type(e).__name__}). Retry in {delay:.1f}s...")
            time.sleep(min(delay, 0.05))  # cap at 50ms for demo

# Flaky function: fails first 2 calls
_n = {'v': 0}
def flaky(prompt):
    _n['v'] += 1
    if _n['v'] <= 2:
        raise ConnectionError("Simulated API timeout")
    return call_llm(prompt)

print("Retry with exponential backoff demo:")
print("-" * 40)
_n['v'] = 0
try:
    r, _, _ = with_retry(flaky, "Classify: Pump P-101 high vibration")
    print(f"  Result: {str(r)[:80]}")
except Exception as e:
    print(f"  Final failure: {e}")

In [ ]:
# CELL 6-C: Circuit breaker simulation
from enum import Enum

class State(Enum):
    CLOSED = 'CLOSED'; OPEN = 'OPEN'; HALF_OPEN = 'HALF_OPEN'

class CircuitBreaker:
    def __init__(self, threshold=3, timeout_s=1.0):
        self.state = State.CLOSED
        self.failures = 0
        self.threshold = threshold
        self.timeout_s = timeout_s
        self.last_fail = None
        self.fast_fails = 0; self.total = 0

    def __call__(self, func, *args, **kwargs):
        self.total += 1
        if self.state == State.OPEN:
            if time.time() - self.last_fail > self.timeout_s:
                self.state = State.HALF_OPEN
                print("    [CB] → HALF_OPEN")
            else:
                self.fast_fails += 1
                raise Exception("Circuit OPEN — fast fail")
        try:
            r = func(*args, **kwargs)
            if self.state == State.HALF_OPEN:
                self.state = State.CLOSED; self.failures = 0
                print("    [CB] → CLOSED (recovered)")
            return r
        except Exception as e:
            self.failures += 1; self.last_fail = time.time()
            if self.failures >= self.threshold:
                self.state = State.OPEN
                print(f"    [CB] → OPEN (failures={self.failures})")
            raise

    def status(self):
        return f"{self.state.value} | fails={self.failures} | fast_fails={self.fast_fails}"

cb = CircuitBreaker(threshold=3, timeout_s=1.0)
_fc = {'n': 0}
def sometimes_fail(p):
    _fc['n'] += 1
    if 4 <= _fc['n'] <= 7:
        raise ConnectionError("API error")
    return call_llm(p)

print("Circuit Breaker — 10 requests:")
print("-" * 55)
for i in range(1, 11):
    try:
        cb(sometimes_fail, f"Ticket {i}")
        print(f"  Request {i:2d}: ✓ | {cb.status()}")
    except Exception as e:
        print(f"  Request {i:2d}: ✗ {str(e)[:30]} | {cb.status()}")
    time.sleep(0.05)

---
## Section 7 — Fine-Tuning with Unsloth

Fine-tuning adapts a pre-trained model to your task or domain.
Three techniques:
1. **Supervised Fine-Tuning (SFT)** — labeled instruction-response pairs
2. **Continued Pretraining** — domain adaptation from raw text
3. **RLHF with DPO** — alignment from human preference pairs

In [ ]:
# CELL 7-A: Fine-tuning decision matrix + setup check
decision = pd.DataFrame([
    {'Scenario': 'Few examples (<100)',         'Prompt Eng': '✅', 'RAG': '❌', 'Fine-Tuning': '❌'},
    {'Scenario': 'Domain knowledge needed',     'Prompt Eng': '⚠️', 'RAG': '✅', 'Fine-Tuning': '⚠️'},
    {'Scenario': 'Specific output format',      'Prompt Eng': '⚠️', 'RAG': '❌', 'Fine-Tuning': '✅'},
    {'Scenario': 'Private data (no cloud)',     'Prompt Eng': '❌', 'RAG': '✅', 'Fine-Tuning': '✅'},
    {'Scenario': 'Low-latency edge device',     'Prompt Eng': '❌', 'RAG': '❌', 'Fine-Tuning': '✅'},
    {'Scenario': '>1000 labeled examples',      'Prompt Eng': '⚠️', 'RAG': '⚠️', 'Fine-Tuning': '✅'},
    {'Scenario': 'Quick iteration needed',      'Prompt Eng': '✅', 'RAG': '⚠️', 'Fine-Tuning': '❌'},
])
print("FINE-TUNING DECISION MATRIX")
print("=" * 65)
print(decision.to_string(index=False))

print("\n\nUnsloth Setup Check")
print("=" * 40)
UNSLOTH_AVAILABLE = False
TRL_AVAILABLE = False
TORCH_AVAILABLE = False

try:
    import unsloth; UNSLOTH_AVAILABLE = True
    print("✓ Unsloth: available")
except ImportError:
    print("ℹ Unsloth: not installed → cells run in SIMULATION mode")
    print("  Install: pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'")

try:
    import torch; TORCH_AVAILABLE = True
    print(f"✓ PyTorch: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)}")
    else:
        print("ℹ CUDA: not available")
except ImportError:
    print("ℹ PyTorch: not installed")

try:
    from trl import SFTTrainer; TRL_AVAILABLE = True
    print("✓ TRL: available")
except ImportError:
    print("ℹ TRL: not installed (pip install trl)")

print(f"\nSimulation mode: {not (UNSLOTH_AVAILABLE and TORCH_AVAILABLE and TRL_AVAILABLE)}")

In [ ]:
# CELL 7-B: Supervised Fine-Tuning (SFT)
def make_sft_dataset(tickets):
    cat_map = {
        'Mechanical': ('Mechanical','High','Schedule maintenance inspection within 24 hours'),
        'Safety':     ('Safety','Critical','Escalate to safety team immediately and isolate area'),
        'Normal':     ('Normal','Low','Complete scheduled PM as planned, close work order'),
        'Complex':    ('Complex','High','Initiate root cause analysis with senior engineer'),
    }
    data = []
    for t in tickets:
        cat, pri, action = cat_map.get(t['category'], ('Unknown','Medium','Review and classify'))
        data.append({
            'instruction': 'Classify this industrial maintenance ticket into category, priority, and action.',
            'input': t['text'],
            'output': f"Category: {cat}\nPriority: {pri}\nAction: {action}"
        })
    return data

sft_data = make_sft_dataset(TICKETS)
print(f"SFT dataset: {len(sft_data)} examples")
print(f"\nSample:")
print(f"  Instruction: {sft_data[0]['instruction']}")
print(f"  Input:       {sft_data[0]['input'][:80]}...")
print(f"  Output:      {sft_data[0]['output']}")

if UNSLOTH_AVAILABLE and TORCH_AVAILABLE and TRL_AVAILABLE and GPU_AVAILABLE:
    print("\n🚀 Real Unsloth training would start here...")
    # (real code omitted — requires GPU + internet)
else:
    print("\n📊 SIMULATION MODE")
    print("LoRA config: r=16, alpha=16, target_modules=[q_proj,v_proj,k_proj,o_proj]")
    print("Training:    epochs=3, batch=2, grad_accum=4, lr=2e-4, optimizer=adamw_8bit")
    print()
    print(f"{'Epoch':<6} {'Step':<6} {'Loss':<10} {'LR'}")
    print("-" * 35)
    for i, loss in enumerate([2.41,2.18,1.95,1.76,1.58,1.43,1.31,1.22,1.14,1.08,1.03,0.99]):
        lr = 2e-4 * (1 - i/12)
        print(f"{i//4+1:<6} {(i%4+1)*5:<6} {loss:<10.4f} {lr:.2e}")
    print("\n✓ Simulated training complete")
    print("  Save: model.save_pretrained_merged('./merged', tokenizer)")
    print("  Push: model.push_to_hub_merged('username/model-name', tokenizer)")

In [ ]:
# CELL 7-C: Continued Pretraining / Domain Adaptation
print("CONTINUED PRETRAINING — DOMAIN ADAPTATION")
print("=" * 50)

raw_corpus = [t['text'] for t in TICKETS] + [
    "Pump P-101 maintenance SOP: Check seal, bearing, impeller every 3 months.",
    "Motor M-202 specifications: 22kW, 1450 RPM, IE3 efficiency class.",
    "Conveyor CV-05 belt tension spec: 175-185 Newton. Measure at center span.",
    "Vibration alarm per ISO 10816: Good<2.8, Satisfactory<7.1, Unsatisfactory<18 mm/s.",
    "Prosedur penggantian bearing: Matikan mesin, pasang LOTO, lepas kopling, ukur clearance.",
    "SOP Inspeksi Pompa: 1) Cek level oli, 2) Ukur vibrasi, 3) Cek suhu bearing, 4) Laporkan.",
]

total_words = sum(len(t.split()) for t in raw_corpus)
print(f"Corpus: {len(raw_corpus)} documents | ~{total_words:,} words estimated")
print(f"\nSample EN: {raw_corpus[0][:80]}...")
print(f"Sample ID: {raw_corpus[-1][:80]}...")

print("\nTraining config (DataCollatorForLanguageModeling):")
print("  mlm = False (causal LM, not masked)")
print("  pad_to_multiple_of = 8")
print("  epochs = 3 (small corpus), lr = 5e-5")

print("\nPerplexity Results (simulated):")
print(f"  {'Model':<35} {'Perplexity':>12} {'Interpretation'}")
print("-" * 65)
print(f"  {'Base LLM (no fine-tuning)':<35} {'28.4':>12}   Higher = less domain familiarity")
print(f"  {'After continued pretraining':<35} {'14.2':>12}   Lower = better domain understanding")
print(f"  {'Improvement':<35} {'50% reduction':>12}   ✓ Significant")

In [ ]:
# CELL 7-D: RLHF with DPO (Direct Preference Optimization)
print("RLHF WITH DPO — ALIGNMENT FROM HUMAN FEEDBACK")
print("=" * 55)

preference_pairs = [
    {'prompt': 'Respond to: Pump P-101 vibration 12mm/s, bearing 85°C',
     'chosen':   'CRITICAL: Pump P-101 shows imminent bearing failure signs. Recommend immediate shutdown. Issue WO-CRIT-001. Notify supervisor.',
     'rejected': 'The pump seems to have some vibration. Maybe check it when you get a chance.'},
    {'prompt': 'Respond to: Filter FLT-03 replacement per scheduled PM',
     'chosen':   'PM completed. Filter FLT-03 replaced per SOP-M-012. P/N: FLT-03-STD. WO-PM-045 closed. Next PM: 90 days.',
     'rejected': 'Filter changed. Done.'},
    {'prompt': 'Respond to: Gas leak H2S 15 ppm near V-301',
     'chosen':   'SAFETY EMERGENCY: H2S 15 ppm exceeds 10 ppm TWA. Area evacuated, LOTO on V-301, safety team notified, monitoring ongoing.',
     'rejected': 'There is a gas smell. Investigate when safe.'},
]

print(f"Preference pairs: {len(preference_pairs)}")
print(f"\nExample:")
print(f"  Prompt:   {preference_pairs[0]['prompt'][:70]}")
print(f"  Chosen:   {preference_pairs[0]['chosen'][:80]}...")
print(f"  Rejected: {preference_pairs[0]['rejected']}")

print("\nDPO Config: beta=0.1, lr=5e-5, epochs=3, batch=1, grad_accum=4")
print("\nSimulated DPO Training:")
print(f"  {'Step':<6} {'Loss':<10} {'Reward Margin':<16} {'KL Div'}")
print("-" * 45)
for step, loss, margin, kl in [(1,0.693,0.02,0.001),(5,0.612,0.15,0.008),
                                 (10,0.534,0.28,0.021),(15,0.471,0.41,0.035),(20,0.423,0.52,0.044)]:
    print(f"  {step:<6} {loss:<10.3f} {margin:<16.3f} {kl:.3f}")

print("\n✓ Reward margin 0.52 → model strongly prefers professional responses")
print("  KL div 0.044 → stayed close to base model (no catastrophic forgetting)")

---
## Section 8 — End-to-End Production Pipeline

Putting it all together: route → classify → draft → monitor.
Runs all 40 tickets through the complete pipeline.

In [ ]:
# CELL 8-A: End-to-end production pipeline
pipeline_tracker = TokenBudgetTracker(budget_usd=1.00, provider=PROVIDER)
pipeline_results = []

CLS_SYS = "Classify ticket.\nCategory: [Mechanical|Safety|Normal|Complex]\nPriority: [Critical|High|Medium|Low]\nReply 2 lines."
DRAFT_SYS = "Write a professional 1-sentence response to this {cat} priority {pri} industrial ticket."

def full_pipeline(ticket):
    # 1. Route
    tier, complexity = route(ticket['text'])[:2]

    # 2. Classify
    t0 = time.time()
    cls_resp, ti1, to1 = call_llm(ticket['text'], system=CLS_SYS, max_tokens=40)
    cls_ms = int((time.time()-t0)*1000)
    pipeline_tracker.record(ticket['id']+'-cls', ticket['category'], ti1, to1, cls_ms)

    predicted_cat, predicted_pri = 'Unknown', 'Medium'
    for line in cls_resp.split('\n'):
        for c in ['Mechanical','Safety','Normal','Complex']:
            if c in line and 'Category' in line: predicted_cat = c
        for p in ['Critical','High','Medium','Low']:
            if p in line and 'Priority' in line: predicted_pri = p

    # 3. Draft
    draft_sys = DRAFT_SYS.format(cat=predicted_cat, pri=predicted_pri)
    t0 = time.time()
    draft, ti2, to2 = call_llm(ticket['text'][:100], system=draft_sys, max_tokens=60)
    draft_ms = int((time.time()-t0)*1000)
    pipeline_tracker.record(ticket['id']+'-draft', ticket['category'], ti2, to2, draft_ms)

    return {'id': ticket['id'], 'actual': ticket['category'],
            'predicted': predicted_cat, 'priority': predicted_pri,
            'tier': tier, 'total_ms': cls_ms+draft_ms,
            'draft': draft[:80]}

print(f"{'ID':<8} {'Actual':<12} {'Predicted':<12} {'Priority':<10} {'Tier':<10} {'ms':>6}")
print("-" * 65)
for t in TICKETS:
    r = full_pipeline(t)
    pipeline_results.append(r)
    print(f"{r['id']:<8} {r['actual']:<12} {r['predicted']:<12} {r['priority']:<10} {r['tier']:<10} {r['total_ms']:>6}")

correct = sum(1 for r in pipeline_results if r['actual'] == r['predicted'])
print(f"\nAccuracy: {correct}/{len(pipeline_results)} ({correct/len(pipeline_results)*100:.0f}%) — mock mode gives approx results")

In [ ]:
# CELL 8-B: Production monitoring dashboard
df_pipe = pd.DataFrame(pipeline_results)
df_recs = pd.DataFrame(pipeline_tracker.records)
s = pipeline_tracker.summary()
all_lat = [r['latency_ms'] for r in pipeline_tracker.records]

print("PRODUCTION MONITORING DASHBOARD")
print("=" * 50)
print(f"  Tickets:       {len(TICKETS)}")
print(f"  LLM calls:     {s['total_requests']}")
print(f"  Tokens in:     {s['total_tokens_in']:,}")
print(f"  Tokens out:    {s['total_tokens_out']:,}")
print(f"  Total cost:    ${s['total_cost_usd']:.4f}")
print(f"  Avg/ticket:    ${s['total_cost_usd']/len(TICKETS):.5f}")
print(f"  p50 latency:   {np.percentile(all_lat,50):.0f}ms")
print(f"  p95 latency:   {np.percentile(all_lat,95):.0f}ms")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Production Monitoring Dashboard — Day 5', fontsize=14, fontweight='bold')

# 1. Cost by category
df_recs.groupby('category')['cost_usd'].sum().plot(
    kind='bar', ax=axes[0,0], color=['#3498db','#e74c3c','#2ecc71','#f39c12'], alpha=0.85)
axes[0,0].set_title('Cost by Category'); axes[0,0].tick_params(axis='x', rotation=30)

# 2. Latency distribution
axes[0,1].hist(all_lat, bins=15, color='#9b59b6', alpha=0.7, edgecolor='black', lw=0.5)
axes[0,1].axvline(np.percentile(all_lat,95), color='red', ls='--', label='p95')
axes[0,1].set_title('Latency Distribution'); axes[0,1].legend()

# 3. Category pie
df_pipe['actual'].value_counts().plot(
    kind='pie', ax=axes[1,0], autopct='%1.0f%%',
    colors=['#3498db','#e74c3c','#2ecc71','#f39c12'])
axes[1,0].set_title('Ticket Mix'); axes[1,0].set_ylabel('')

# 4. Cumulative cost
axes[1,1].plot([r['cumulative_cost'] for r in pipeline_tracker.records], color='#1abc9c', lw=2)
axes[1,1].axhline(pipeline_tracker.budget_usd*0.8, color='orange', ls='--', alpha=0.7, label='80% warning')
axes[1,1].set_title('Cumulative Cost'); axes[1,1].set_xlabel('LLM call #'); axes[1,1].legend()

plt.tight_layout()
plt.show()
print("✓ Dashboard complete")

---
## Section 9 — Reflection & Program Wrap-up

---

### 🎓 Five-Day Learning Journey

| Day | Topic | Key Achievement |
|-----|-------|-----------------|
| Day 1 | Text Mining & Clustering | K-Means on 40 tickets; silhouette score 0.027 |
| Day 2 | LLM APIs & Prompt Engineering | 90%+ categorization accuracy; structured JSON output |
| Day 3 | RAG Pipeline | Grounded answers from 30-document knowledge base |
| Day 4 | LLM Orchestration | Autonomous agent with LangChain + LangGraph + HITL |
| Day 5 | Deployment & Production | Full production pipeline: deploy, monitor, fine-tune |

---

### ✅ Production Deployment Checklist

**Infrastructure**
- [ ] Deployment mode chosen (cloud API / vLLM / Ollama)
- [ ] Hardware sized for model + quantization level
- [ ] API key management (environment variables, secrets manager)
- [ ] Rate limits configured

**Reliability**
- [ ] Retry with exponential backoff
- [ ] Circuit breaker (threshold=3, timeout=60s)
- [ ] Fallback chain: primary → secondary → rule-based
- [ ] Health check endpoint

**Cost & Performance**
- [ ] TokenBudgetTracker deployed
- [ ] Cost-aware routing configured
- [ ] SLA targets defined (p95 < 2000ms)
- [ ] Semantic cache (target: 30%+ hit rate)

**Monitoring**
- [ ] p50/p95/p99 latency dashboarded
- [ ] Cost per request + daily budget tracked
- [ ] Error rate alerting configured
- [ ] LangSmith tracing enabled

**Security**
- [ ] PII masking before cloud API calls
- [ ] Prompt injection detection
- [ ] Audit log of all LLM calls
- [ ] Output validation

**Fine-Tuning (optional)**
- [ ] Baseline evaluation established
- [ ] Dataset curated (>500 examples for SFT)
- [ ] Fine-tuned model evaluated on held-out set
- [ ] Adapter pushed to HuggingFace Hub

---

### 🚀 Next Steps & Resources

**Immediate actions:**
1. Try Ollama locally: `curl -fsSL https://ollama.ai/install.sh | sh && ollama pull llama3.2`
2. Run your first Unsloth fine-tune on Google Colab (free T4 GPU)
3. Deploy this pipeline to a cloud VM with Docker
4. Enable LangSmith tracing on your Day 4 agent

**Key resources:**
- **vLLM**: https://docs.vllm.ai
- **Ollama**: https://ollama.ai
- **Unsloth**: https://github.com/unslothai/unsloth
- **TRL (SFT/DPO)**: https://huggingface.co/docs/trl
- **LangSmith**: https://smith.langchain.com

**Papers:** PagedAttention (2023) · DPO (2023) · QLoRA (2023) · LoRA (2021)

---

*Thank you for five incredible nights of learning. You now have the complete stack —
from raw text to production deployment — to build real industrial AI systems.*

*— Industrial AI & LLM Training Program*